In [1]:
print("Hello! My Texas Electricity Project is ready.")

Hello! My Texas Electricity Project is ready.


In [3]:
import pandas as pd

df = pd.read_excel("../data/raw/Native_Load_2025.xlsx")

print(df.shape)
print(df.head())
df.info()


(8760, 10)
        Hour Ending         COAST         EAST        FWEST        NORTH  \
0  01/01/2025 01:00  10538.553193  1545.465863  7090.619810  1388.565588   
1  01/01/2025 02:00  10477.912860  1553.994477  7040.460607  1390.392647   
2  01/01/2025 03:00  10363.015361  1534.575620  7028.741719  1390.634577   
3  01/01/2025 04:00  10259.984795  1548.084385  6992.734978  1402.423965   
4  01/01/2025 05:00  10332.692971  1585.772017  7026.066195  1425.900669   

          NCENT        SOUTH        SCENT         WEST         ERCOT  
0  11964.866264  3012.769998  6447.297871  1235.721317  43223.859905  
1  12064.919506  2962.803597  6382.650858  1249.397659  43122.532212  
2  12040.262276  2981.433181  6320.119991  1259.729121  42918.511847  
3  12055.424000  2970.761172  6277.132219  1274.238069  42780.783583  
4  12307.605237  2975.631819  6298.522746  1301.043053  43253.234707  
<class 'pandas.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 10 columns):
 #   Colum

In [ ]:
# Check missing values
print("Missing values:")
print(df.isnull().sum())

# Check duplicates
print("\nDuplicate rows:")
print(df.duplicated().sum())

# Check data types
print("\nData types:")
print(df.dtypes)

Missing values:
Hour Ending    0
COAST          0
EAST           0
FWEST          0
NORTH          0
NCENT          0
SOUTH          0
SCENT          0
WEST           0
ERCOT          0
dtype: int64

Duplicate rows:
0

Data types:
Hour Ending        str
COAST          float64
EAST           float64
FWEST          float64
NORTH          float64
NCENT          float64
SOUTH          float64
SCENT          float64
WEST           float64
ERCOT          float64
dtype: object


In [5]:
# Inspect first and last timestamps
print("First 5 timestamps:")
print(df["Hour Ending"].head())

print("\nLast 5 timestamps:")
print(df["Hour Ending"].tail())

# Identify special timestamp formats
print("\n24:00 timestamps:")
print(df["Hour Ending"].str.contains("24:00").sum())

print("\nDST timestamps:")
print(df["Hour Ending"].str.contains("DST", case=False).sum())

First 5 timestamps:
0    01/01/2025 01:00
1    01/01/2025 02:00
2    01/01/2025 03:00
3    01/01/2025 04:00
4    01/01/2025 05:00
Name: Hour Ending, dtype: str

Last 5 timestamps:
8755    12/31/2025 20:00
8756    12/31/2025 21:00
8757    12/31/2025 22:00
8758    12/31/2025 23:00
8759    12/31/2025 24:00
Name: Hour Ending, dtype: str

24:00 timestamps:
365

DST timestamps:
1


In [6]:
# Find the DST record
dst_rows = df[
    df["Hour Ending"].str.contains("DST", case=False)
]

print("DST record:")
print(dst_rows["Hour Ending"])

# Display surrounding timestamps
for index in dst_rows.index:
    print("\nSurrounding timestamps:")
    print(df.loc[index-3:index+3, "Hour Ending"])

DST record:
7321    11/02/2025 02:00 DST
Name: Hour Ending, dtype: str

Surrounding timestamps:
7318        11/01/2025 24:00
7319        11/02/2025 01:00
7320        11/02/2025 02:00
7321    11/02/2025 02:00 DST
7322        11/02/2025 03:00
7323        11/02/2025 04:00
7324        11/02/2025 05:00
Name: Hour Ending, dtype: str


In [7]:
import pandas as pd

# Preserve original data
df_clean = df.copy()

# Remove DST suffix for parsing
timestamp_text = (
    df_clean["Hour Ending"]
    .str.replace(" DST", "", regex=False)
)

# Identify 24:00 records
is_24 = timestamp_text.str.endswith("24:00")

# Replace 24:00 with 00:00 for parsing
timestamp_text = timestamp_text.str.replace(
    "24:00", "00:00", regex=False
)

# Convert to datetime
df_clean["hour_ending_local"] = pd.to_datetime(
    timestamp_text,
    format="%m/%d/%Y %H:%M"
)

# Move 24:00 records to the next day
df_clean.loc[is_24, "hour_ending_local"] += pd.Timedelta(days=1)

print(df_clean[["Hour Ending", "hour_ending_local"]].head())

print("\n24:00 conversion example:")
print(
    df_clean.loc[
        df["Hour Ending"].str.contains("24:00"),
        ["Hour Ending", "hour_ending_local"]
    ].head()
)

        Hour Ending   hour_ending_local
0  01/01/2025 01:00 2025-01-01 01:00:00
1  01/01/2025 02:00 2025-01-01 02:00:00
2  01/01/2025 03:00 2025-01-01 03:00:00
3  01/01/2025 04:00 2025-01-01 04:00:00
4  01/01/2025 05:00 2025-01-01 05:00:00

24:00 conversion example:
          Hour Ending hour_ending_local
23   01/01/2025 24:00        2025-01-02
47   01/02/2025 24:00        2025-01-03
71   01/03/2025 24:00        2025-01-04
95   01/04/2025 24:00        2025-01-05
119  01/05/2025 24:00        2025-01-06


In [8]:
# Calculate differences between consecutive timestamps
time_diff = df_clean["hour_ending_local"].diff()

print("Time differences:")
print(time_diff.value_counts())

# Find timestamps that are not exactly one hour apart
unusual_rows = df_clean.loc[
    time_diff.notna() & (time_diff != pd.Timedelta(hours=1)),
    ["Hour Ending", "hour_ending_local"]
]

print("\nUnusual timestamp rows:")
print(unusual_rows)

Time differences:
hour_ending_local
0 days 01:00:00    8757
0 days 02:00:00       1
0 days 00:00:00       1
Name: count, dtype: int64

Unusual timestamp rows:
               Hour Ending   hour_ending_local
1610      03/09/2025 04:00 2025-03-09 04:00:00
7321  11/02/2025 02:00 DST 2025-11-02 02:00:00


In [10]:
# Build a continuous UTC hourly timeline
df_clean["hour_ending_utc"] = pd.date_range(
    start="2025-01-01 07:00:00",
    periods=len(df_clean),
    freq="h",
    tz="UTC"
)

# Convert UTC to Central Time
df_clean["hour_ending_central"] = (
    df_clean["hour_ending_utc"]
    .dt.tz_convert("America/Chicago")
)

# Examine the DST transition
print(
    df_clean[
        ["Hour Ending", "hour_ending_central", "hour_ending_utc"]
    ].iloc[7318:7324].to_string(index=False)
)

         Hour Ending       hour_ending_central           hour_ending_utc
    11/01/2025 24:00 2025-11-02 00:00:00-05:00 2025-11-02 05:00:00+00:00
    11/02/2025 01:00 2025-11-02 01:00:00-05:00 2025-11-02 06:00:00+00:00
    11/02/2025 02:00 2025-11-02 01:00:00-06:00 2025-11-02 07:00:00+00:00
11/02/2025 02:00 DST 2025-11-02 02:00:00-06:00 2025-11-02 08:00:00+00:00
    11/02/2025 03:00 2025-11-02 03:00:00-06:00 2025-11-02 09:00:00+00:00
    11/02/2025 04:00 2025-11-02 04:00:00-06:00 2025-11-02 10:00:00+00:00


In [11]:
print("Total records:", len(df_clean))

print(
    "Duplicate UTC timestamps:",
    df_clean["hour_ending_utc"].duplicated().sum()
)

print(
    "UTC hourly differences:"
)

print(
    df_clean["hour_ending_utc"]
    .diff()
    .value_counts()
)

Total records: 8760
Duplicate UTC timestamps: 0
UTC hourly differences:
hour_ending_utc
0 days 01:00:00    8759
Name: count, dtype: int64


In [12]:
# Inspect the spring DST transition
spring_rows = df_clean[
    df_clean["Hour Ending"].str.startswith("03/09/2025")
]

print(
    spring_rows[
        ["Hour Ending", "hour_ending_central", "hour_ending_utc"]
    ].head(7).to_string(index=False)
)

     Hour Ending       hour_ending_central           hour_ending_utc
03/09/2025 01:00 2025-03-09 01:00:00-06:00 2025-03-09 07:00:00+00:00
03/09/2025 02:00 2025-03-09 03:00:00-05:00 2025-03-09 08:00:00+00:00
03/09/2025 04:00 2025-03-09 04:00:00-05:00 2025-03-09 09:00:00+00:00
03/09/2025 05:00 2025-03-09 05:00:00-05:00 2025-03-09 10:00:00+00:00
03/09/2025 06:00 2025-03-09 06:00:00-05:00 2025-03-09 11:00:00+00:00
03/09/2025 07:00 2025-03-09 07:00:00-05:00 2025-03-09 12:00:00+00:00
03/09/2025 08:00 2025-03-09 08:00:00-05:00 2025-03-09 13:00:00+00:00


In [13]:
# Define the eight ERCOT weather zones
zones = [
    "COAST", "EAST", "FWEST", "NORTH",
    "NCENT", "SOUTH", "SCENT", "WEST"
]

# Calculate total demand from the zones
df_clean["calculated_total"] = df_clean[zones].sum(axis=1)

# Compare against reported ERCOT demand
df_clean["total_difference"] = (
    df_clean["ERCOT"] - df_clean["calculated_total"]
)

print("Maximum absolute difference:")
print(df_clean["total_difference"].abs().max())

print("\nAverage absolute difference:")
print(df_clean["total_difference"].abs().mean())

print("\nRows with difference greater than 1 MW:")
print((df_clean["total_difference"].abs() > 1).sum())

print("\nNegative electricity values:")
print((df_clean[zones + ["ERCOT"]] < 0).sum())

Maximum absolute difference:
3.0000010156072676e-06

Average absolute difference:
5.642711821814348e-07

Rows with difference greater than 1 MW:
0

Negative electricity values:
COAST    0
EAST     0
FWEST    0
NORTH    0
NCENT    0
SOUTH    0
SCENT    0
WEST     0
ERCOT    0
dtype: int64


In [14]:
print("Negative values:")
print((df_clean[zones + ["ERCOT"]] < 0).sum())

print("\nZero values:")
print((df_clean[zones + ["ERCOT"]] == 0).sum())

Negative values:
COAST    0
EAST     0
FWEST    0
NORTH    0
NCENT    0
SOUTH    0
SCENT    0
WEST     0
ERCOT    0
dtype: int64

Zero values:
COAST    0
EAST     0
FWEST    0
NORTH    0
NCENT    0
SOUTH    0
SCENT    0
WEST     0
ERCOT    0
dtype: int64


In [15]:
print(df_clean[zones + ["ERCOT"]].describe().round(2))

print("\nHighest demand:")
print(df_clean["ERCOT"].max())

print("\nLowest demand:")
print(df_clean["ERCOT"].min())

print("\nAverage demand:")
print(df_clean["ERCOT"].mean().round(2))

          COAST     EAST    FWEST    NORTH     NCENT    SOUTH     SCENT  \
count   8760.00  8760.00  8760.00  8760.00   8760.00  8760.00   8760.00   
mean   14745.71  1817.98  7478.22  1689.52  15306.62  4434.74   8877.04   
std     2833.64   426.64   418.23   319.31   3736.95   862.13   2048.06   
min     9997.44  1054.63  6042.46   918.20   9075.37  2738.22   5409.31   
25%    12542.91  1502.64  7182.14  1449.48  12581.49  3784.42   7297.03   
50%    14193.34  1716.93  7472.26  1668.15  14386.48  4261.82   8422.89   
75%    16528.79  2077.60  7755.99  1889.59  17393.06  5045.50  10183.91   
max    23006.70  3696.72  8773.08  2732.09  27531.76  7155.62  14799.33   

          WEST     ERCOT  
count  8760.00   8760.00  
mean   1384.60  55734.43  
std     238.83  10140.33  
min     927.34  38362.79  
25%    1206.37  48151.65  
50%    1330.52  53407.20  
75%    1530.42  61998.62  
max    2116.51  83679.00  

Highest demand:
83679.001697

Lowest demand:
38362.786561

Average demand:
55734

In [16]:
# Extract time features from the original ERCOT reporting date
reporting_date = pd.to_datetime(
    df_clean["Hour Ending"].str[:10],
    format="%m/%d/%Y"
)

df_clean["date"] = reporting_date
df_clean["year"] = reporting_date.dt.year
df_clean["month"] = reporting_date.dt.month
df_clean["day_of_week"] = reporting_date.dt.day_name()

# Use actual Central clock hour
df_clean["hour"] = df_clean["hour_ending_central"].dt.hour

print(
    df_clean[
        ["date", "year", "month", "day_of_week", "hour"]
    ].head()
)

        date  year  month day_of_week  hour
0 2025-01-01  2025      1   Wednesday     1
1 2025-01-01  2025      1   Wednesday     2
2 2025-01-01  2025      1   Wednesday     3
3 2025-01-01  2025      1   Wednesday     4
4 2025-01-01  2025      1   Wednesday     5


In [17]:
df_zones = df_clean.melt(
    id_vars=[
        "hour_ending_utc",
        "date",
        "month",
        "day_of_week",
        "hour"
    ],
    value_vars=zones,
    var_name="zone",
    value_name="demand_mw"
)

print("Regional dataset shape:", df_zones.shape)
print(df_zones.head())
print("\nRecords per zone:")
print(df_zones["zone"].value_counts())

Regional dataset shape: (70080, 7)
            hour_ending_utc       date  month day_of_week  hour   zone  \
0 2025-01-01 07:00:00+00:00 2025-01-01      1   Wednesday     1  COAST   
1 2025-01-01 08:00:00+00:00 2025-01-01      1   Wednesday     2  COAST   
2 2025-01-01 09:00:00+00:00 2025-01-01      1   Wednesday     3  COAST   
3 2025-01-01 10:00:00+00:00 2025-01-01      1   Wednesday     4  COAST   
4 2025-01-01 11:00:00+00:00 2025-01-01      1   Wednesday     5  COAST   

      demand_mw  
0  10538.553193  
1  10477.912860  
2  10363.015361  
3  10259.984795  
4  10332.692971  

Records per zone:
zone
COAST    8760
EAST     8760
FWEST    8760
NORTH    8760
NCENT    8760
SOUTH    8760
SCENT    8760
WEST     8760
Name: count, dtype: int64


In [18]:
from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

df_statewide = df_clean[
    [
        "hour_ending_utc",
        "date",
        "month",
        "day_of_week",
        "hour",
        "ERCOT"
    ]
].rename(columns={"ERCOT": "demand_mw"})

df_statewide.to_csv(
    output_dir / "ercot_statewide_2025.csv",
    index=False
)

df_zones.to_csv(
    output_dir / "ercot_zones_2025.csv",
    index=False
)

print("Statewide records:", len(df_statewide))
print("Regional records:", len(df_zones))
print("Files exported successfully!")

Statewide records: 8760
Regional records: 70080
Files exported successfully!
